In [1]:
!pip install sacrebleu
!pip install transformers
!pip install accelerate
!pip install evaluate

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
import os
direc = "/dss/dsshome1/09/go34jos2"
train_src_file= os.path.join(direc,"train.de-hsb.de")
train_tgt_file = os.path.join(direc, "train.de-hsb.hsb")
dev_src_file =os.path.join(direc,"dev.de-hsb.de")
dev_tgt_file = os.path.join(direc, "dev.de-hsb.hsb")

In [3]:
import random
import re
import torch
from tqdm import tqdm
import evaluate
from transformers import AutoTokenizer, AutoModelForCausalLM
with open(dev_src_file,encoding='utf-8') as f:
  val_s =[line.strip() for line in f.readlines() if line.strip()]
with open(dev_tgt_file,encoding='utf-8') as f:
  val_t = [line.strip() for line in f.readlines() if line.strip()]
with open(train_src_file, encoding='utf-8' ) as f:
  train_s = [line.strip() for line in f.readlines() if line.strip()]
with open(train_tgt_file,encoding='utf-8') as f:
  train_t = [line.strip() for line in f.readlines() if line.strip()]
random.seed(1000)
#3shot
shot_indices= random.sample(range(len(train_s)),3)
#prompt suggested by NRC
instruction_text = "Translate the following German text to Upper Sorbian.\nPut it in this format <hsb> Upper Sorbian translation </hsb>."
base_messages= [{"role": "system", "content": instruction_text}]
for idx in shot_indices:
  src_ex =train_s[idx]
  tgt_ex =train_t[idx]
  base_messages.append({"role": "user", "content": f"<deu> {src_ex} </deu>"})
  base_messages.append({"role": "assistant", "content": f"<hsb> {tgt_ex} </hsb>"})

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
model_name = "Qwen/Qwen3-0.6B"
tokenizer =AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto"
)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
if tokenizer.chat_template is None:
  #use chatml format
  tokenizer.chat_template = "{% if not add_generation_prompt is defined %}{% set add_generation_prompt = false %}{% endif %}{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
def extract_translation(text):
  match = re.search(r"<hsb>\s*(.*?)\s*</hsb>",text, re.DOTALL)
  if match:
    return match.group(1).strip()
  return text.replace("<hsb>", "").replace("</hsb>", "").strip()
predictions =[]
batch_size =16
for i in tqdm(range(0, len(val_s), batch_size)):
  batch_src= val_s[i:i+batch_size]
  batch_prompts = []
  for src in batch_src:
    current_messages=base_messages.copy()
    current_messages.append({"role": "user", "content": f"<deu> {src} </deu>"})
    text= tokenizer.apply_chat_template(current_messages,tokenize=False, add_generation_prompt=True)
    batch_prompts.append(text)
  inputs = tokenizer(batch_prompts, return_tensors="pt",padding=True, truncation=True).to(model.device)
  with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id
        
    )
  generated_ids =[out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
  raw_preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
  clean_preds = [extract_translation(p) for p in raw_preds]
  predictions.extend(clean_preds)

100%|██████████| 250/250 [1:13:29<00:00, 17.64s/it]


In [5]:
sacrebleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")
bleu_result =sacrebleu.compute(predictions=predictions, references=[[r] for r in val_t])
chrf_result = chrf.compute(predictions=predictions, references=[[r] for r in val_t], word_order=2)
print(f"SacreBLEU: {bleu_result['score']:.2f}")
print(f"chrF++:    {chrf_result['score']:.2f}")

SacreBLEU: 0.01
chrF++:    1.85
